# Aula 07 - Notebook: Validade de Argumentos e Inferência Lógica na Segurança de Processos
## SCADA-Core Automática — Grupo 04: Classificação e Seleção de Grãos por Visão Computacional

Neste notebook implementamos a arquitetura do **Provador Dedutivo Formal**, um **SAT Solver com Algoritmo DPLL (Davis-Putnam-Logemann-Loveland)** e um **Verificador de Consistência Axiomática** para certificar matematicamente a segurança da planta de classificação de grãos:
1. **Verificação Exaustiva por Tabela-Verdade** ($2^n$ estados operacionais).
2. **SAT Solver Clássico com Propagação Unitária e Eliminação de Literais Puros (DPLL)**.
3. **Passos de Inferência Formal Passo a Passo** (Modus Ponens, Modus Tollens, Silogismo Hipotético e Resolução).
4. **Prova dos 5 Teoremas Fundamentais de Segurança da Planta**:
   - *Teorema 1:* Desarme Universal da Alimentação em Falha Crítica / Trip Geral.
   - *Teorema 2:* Inibição de Ejeção Pneumática Cega sob Queda de Pressão de Ar.
   - *Teorema 3:* Exclusão Mútua Tripla das Categorias de Grãos ($p_A, p_B, p_C$).
   - *Teorema 4:* Proteção contra Partida em Sobrecarga Elétrica do Motor.
   - *Teorema 5:* Teorema da Qualidade Assegurada (Grãos Cat A são imunes a defeitos).
5. **Módulo de Análise e Detecção de Falácias Formais** na automação industrial com listagem exaustiva de contraexemplos.


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionários em tabela ASCII pura alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import itertools
from typing import List, Dict, Callable, Any, Set, Tuple, Optional

class DPLLSolver:
    """Implementação do Algoritmo DPLL para Resolução de Satisfatibilidade Booleana (SAT)."""
    
    @staticmethod
    def propagacao_unitaria(clausulas: List[Set[int]], atribuicao: Dict[int, bool]) -> Tuple[List[Set[int]], Dict[int, bool], bool]:
        """Aplica propagação de cláusulas unitárias recursivamente."""
        clausulas_atuais = [set(c) for c in clausulas]
        modificado = True
        
        while modificado:
            modificado = False
            unitarias = [c for c in clausulas_atuais if len(c) == 1]
            for u in unitarias:
                lit = next(iter(u))
                var = abs(lit)
                val = (lit > 0)
                if var in atribuicao and atribuicao[var] != val:
                    return [], atribuicao, False # Conflito / Contradição
                atribuicao[var] = val
                
                novas_clausulas = []
                for c in clausulas_atuais:
                    if lit in c:
                        continue # Cláusula satisfeita
                    if -lit in c:
                        c_reduzida = c - {-lit}
                        if not c_reduzida:
                            return [], atribuicao, False # Cláusula vazia -> Contradição
                        novas_clausulas.append(c_reduzida)
                    else:
                        novas_clausulas.append(c)
                clausulas_atuais = novas_clausulas
                modificado = True
                break
        return clausulas_atuais, atribuicao, True

    @classmethod
    def resolver(cls, clausulas: List[Set[int]], atribuicao: Optional[Dict[int, bool]] = None) -> Tuple[bool, Dict[int, bool]]:
        """Executa a busca recursiva DPLL com backtracking."""
        if atribuicao is None:
            atribuicao = {}
        
        clausulas_simplificadas, atribuicao, consistente = cls.propagacao_unitaria(clausulas, dict(atribuicao))
        if not consistente:
            return False, {}
        if not clausulas_simplificadas:
            return True, atribuicao
        
        var_escolhida = abs(next(iter(clausulas_simplificadas[0])))
        
        atrib_true = dict(atribuicao)
        atrib_true[var_escolhida] = True
        sat_t, sol_t = cls.resolver(clausulas_simplificadas + [{var_escolhida}], atrib_true)
        if sat_t:
            return True, sol_t
            
        atrib_false = dict(atribuicao)
        atrib_false[var_escolhida] = False
        sat_f, sol_f = cls.resolver(clausulas_simplificadas + [{-var_escolhida}], atrib_false)
        if sat_f:
            return True, sol_f
            
        return False, {}

class ProvadorDedutivoFormal:
    """Motor algorítmico de inferência formal para validação de matrizes de segurança industrial."""
    
    @staticmethod
    def verificar_tabela_verdade(
        nome_teorema: str,
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """Verifica a validade do argumento P1, P2, ..., Pk |- C por varredura exaustiva de 2^n estados."""
        n = len(variaveis)
        total_estados = 2 ** n
        linhas_premissas_true = 0
        linhas_conclusao_true = 0
        contraexemplos = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(p(env) for p in premissas):
                linhas_premissas_true += 1
                if conclusao(env):
                    linhas_conclusao_true += 1
                else:
                    contraexemplos.append(env)
                    
        valido = (linhas_premissas_true > 0) and (linhas_premissas_true == linhas_conclusao_true)
        
        return {
            "Teorema": nome_teorema,
            "Variáveis (n)": n,
            "Total Estados (2^n)": total_estados,
            "Estados Válidos (Premissas True)": linhas_premissas_true,
            "Estados Conformes": linhas_conclusao_true,
            "Válido": valido,
            "Status": "TEOREMA VÁLIDO (SEGURO)" if valido else "FALÁCIA / INVÁLIDO",
            "Contraexemplos": contraexemplos
        }

    @staticmethod
    def provar_por_refutacao_sat(
        nome_teorema: str,
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """Método de Prova por Refutação (SAT Solver): P1..Pk |- C é válido sse {P1..Pk, NOT C} é insatisfatível."""
        n = len(variaveis)
        modelos_violacao = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(p(env) for p in premissas) and not conclusao(env):
                modelos_violacao.append(env)
                
        insatisfativel = len(modelos_violacao) == 0
        return {
            "Teorema": nome_teorema,
            "Insatisfatível": insatisfativel,
            "Modelos de Falha": len(modelos_violacao),
            "Resultado SAT": "CONTRADIÇÃO COMPROVADA (ESTADO PROIBIDO INALCANÇÁVEL)" if insatisfativel else "MODELO DE FALHA ENCONTRADO",
            "Lista Modelos": modelos_violacao
        }

    @staticmethod
    def verificar_consistencia_axiomatica(nome_base: str, variaveis: List[str], axiomas: List[Callable[[Dict[str, bool]], bool]]) -> Dict[str, Any]:
        """Verifica se o conjunto de premissas/axiomas da planta não possui contradição interna."""
        n = len(variaveis)
        estados_satisfativeis = 0
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(ax(env) for ax in axiomas):
                estados_satisfativeis += 1
        consistente = estados_satisfativeis > 0
        return {
            "Base de Axiomas": nome_base,
            "Estados Satisfatíveis": estados_satisfativeis,
            "Consistente": consistente,
            "Diagnóstico": "BASE CONSISTENTE (LIVRE DE PARADOXOS)" if consistente else "BASE CONTRADITÓRIA"
        }

print("[OK] ProvadorDedutivoFormal e DPLL SAT Solver inicializados com sucesso!")


## 1. Verificação de Consistência Axiomática da Planta
Antes de provar teoremas, provamos matematicamente que o conjunto de definições de intertravamento do SCADA-Core não contém paradoxos ou contradições internas ($\Gamma 
ot
dash ot$).


In [ ]:
# Variáveis fundamentais da matriz de segurança da planta
vars_base = ['p_EMERG', 'p_JI201', 'p_PAL601', 'p_NC703', 'p_KSA401', 'p_MOV201', 'p_NB101', 'c_PERM', 'c_ALIM', 'c_FY603', 'p_C', 'p_POS603']

# Axiomas da Planta de Grãos
ax1_perm = lambda e: e['c_PERM'] == (not e['p_EMERG'] and not e['p_JI201'] and not e['p_PAL601'] and not e['p_NC703'] and e['p_KSA401'])
ax2_alim = lambda e: e['c_ALIM'] == (e['c_PERM'] and e['p_MOV201'] and not e['p_NB101'])
ax3_ejetor = lambda e: e['c_FY603'] == (e['p_C'] and e['p_POS603'] and not e['p_PAL601'])

axiomas_planta = [ax1_perm, ax2_alim, ax3_ejetor]
res_consistencia = ProvadorDedutivoFormal.verificar_consistencia_axiomatica("Matriz de Intertravamento SCADA-Core", vars_base, axiomas_planta)

print("=== VERIFICAÇÃO DE CONSISTÊNCIA AXIOMÁTICA ===")
print(f"Base Analisada: {res_consistencia['Base de Axiomas']}")
print(f"Estados Operacionais Válidos: {res_consistencia['Estados Satisfatíveis']} de {2**len(vars_base)}")
print(f"Diagnóstico Formal: {res_consistencia['Diagnóstico']}")

assert res_consistencia['Consistente'] is True


## 2. Teoremas 1 e 2: Intertravamento Crítico e Proteção Pneumática
* **Teorema 1:** Garantia formal de desarme da alimentação ($c_{	ext{ALIM}} = 0$) sob qualquer Trip Geral ($	ext{Trip}_{	ext{GERAL}} \implies 
eg c_{	ext{ALIM}}$).
* **Teorema 2:** Inibição do ejetor pneumático FY-603 sob despressurização de linha ($p_{	ext{PAL601}} \implies 
eg c_{	ext{FY603}}$).


In [ ]:
# ------------------------------------------------------------------------------
# TEOREMA 1: Desarme Geral da Alimentação (Trip Crítico)
# ------------------------------------------------------------------------------
vars_t1 = ['p_EMERG', 'p_JI201', 'p_PAL601', 'p_NC703', 'p_KSA401', 'p_MOV201', 'p_NB101', 'c_PERM', 'c_ALIM']

p1_perm = lambda e: e['c_PERM'] == (not e['p_EMERG'] and not e['p_JI201'] and not e['p_PAL601'] and not e['p_NC703'] and e['p_KSA401'])
p2_alim = lambda e: e['c_ALIM'] == (e['c_PERM'] and e['p_MOV201'] and not e['p_NB101'])

def concl_t1(e: Dict[str, bool]) -> bool:
    trip_geral = e['p_EMERG'] or e['p_JI201'] or e['p_PAL601'] or e['p_NC703'] or not e['p_KSA401']
    if trip_geral:
        return not e['c_ALIM']
    return True

res_t1_tab = ProvadorDedutivoFormal.verificar_tabela_verdade("Teorema 1: Desarme de Alimentação", vars_t1, [p1_perm, p2_alim], concl_t1)
res_t1_sat = ProvadorDedutivoFormal.provar_por_refutacao_sat("Teorema 1: Desarme de Alimentação", vars_t1, [p1_perm, p2_alim], concl_t1)

# ------------------------------------------------------------------------------
# TEOREMA 2: Anti-Ejeção Cega sob Queda de Pressão Pneumática
# ------------------------------------------------------------------------------
vars_t2 = ['p_C', 'p_POS603', 'p_PAL601', 'c_FY603']
p_ejetor = lambda e: e['c_FY603'] == (e['p_C'] and e['p_POS603'] and not e['p_PAL601'])
concl_t2 = lambda e: (not e['p_PAL601']) or (not e['c_FY603'])

res_t2_tab = ProvadorDedutivoFormal.verificar_tabela_verdade("Teorema 2: Anti-Ejeção Cega", vars_t2, [p_ejetor], concl_t2)
res_t2_sat = ProvadorDedutivoFormal.provar_por_refutacao_sat("Teorema 2: Anti-Ejeção Cega", vars_t2, [p_ejetor], concl_t2)

relatorio_t1_t2 = [
    {"Teorema": res_t1_tab['Teorema'], "Estados (2^n)": res_t1_tab['Total Estados (2^n)'], "Linhas Consistentes": res_t1_tab['Estados Válidos (Premissas True)'], "Status Tabela": res_t1_tab['Status'], "Status SAT": res_t1_sat['Resultado SAT']},
    {"Teorema": res_t2_tab['Teorema'], "Estados (2^n)": res_t2_tab['Total Estados (2^n)'], "Linhas Consistentes": res_t2_tab['Estados Válidos (Premissas True)'], "Status Tabela": res_t2_tab['Status'], "Status SAT": res_t2_sat['Resultado SAT']}
]
print("=== RESULTADOS DAS PROVAS FORMAIS (TEOREMAS 1 E 2) ===")
print(formatar_tabela(relatorio_t1_t2))

assert res_t1_tab['Válido'] is True
assert res_t1_sat['Insatisfatível'] is True
assert res_t2_tab['Válido'] is True
assert res_t2_sat['Insatisfatível'] is True


## 3. Teoremas 3, 4 e 5: Exclusão Mútua, Proteção Elétrica e Imunidade de Qualidade
* **Teorema 3:** Exclusão Mútua Absoluta ($
eg(p_A \land p_B) \land 
eg(p_A \land p_C) \land 
eg(p_B \land p_C)$).
* **Teorema 4:** Proteção do Motor da Esteira contra Partida em Sobrecarga Elétrica ($
eg (c_{	ext{ESTEIRA}} \land p_{	ext{JI201}})$).
* **Teorema 5:** Teorema da Qualidade Assegurada: Grãos de Categoria A nunca possuem defeitos ($p_A \implies 
eg (p_{	ext{CV107}} \lor p_{	ext{CV108}} \lor p_{	ext{CV109}})$).


In [ ]:
# ------------------------------------------------------------------------------
# TEOREMA 3: Exclusão Mútua Tripla das Categorias de Grãos
# ------------------------------------------------------------------------------
vars_t3 = ['p_CV_IDEAL', 'p_CV_DEFEITO', 'p_A', 'p_B', 'p_C']
p_cat_a = lambda e: e['p_A'] == (e['p_CV_IDEAL'] and not e['p_CV_DEFEITO'])
p_cat_c = lambda e: e['p_C'] == (e['p_CV_DEFEITO'])
p_cat_b = lambda e: e['p_B'] == (not e['p_A'] and not e['p_C'])
concl_t3 = lambda e: not (e['p_A'] and e['p_B']) and not (e['p_A'] and e['p_C']) and not (e['p_B'] and e['p_C'])

res_t3_tab = ProvadorDedutivoFormal.verificar_tabela_verdade("Teorema 3: Exclusão Mútua de Grãos", vars_t3, [p_cat_a, p_cat_c, p_cat_b], concl_t3)

# ------------------------------------------------------------------------------
# TEOREMA 4: Proteção Elétrica do Motor contra Sobrecarga
# ------------------------------------------------------------------------------
vars_t4 = ['p_JI201', 'c_PERM', 'c_ESTEIRA']
p_perm_eletrica = lambda e: (not e['p_JI201']) if e['c_PERM'] else True
p_cmd_esteira   = lambda e: (not e['c_ESTEIRA']) if not e['c_PERM'] else True
concl_t4        = lambda e: not (e['c_ESTEIRA'] and e['p_JI201'])

res_t4_tab = ProvadorDedutivoFormal.verificar_tabela_verdade("Teorema 4: Proteção Elétrica Motor", vars_t4, [p_perm_eletrica, p_cmd_esteira], concl_t4)

# ------------------------------------------------------------------------------
# TEOREMA 5: Qualidade Assegurada Categoria A
# ------------------------------------------------------------------------------
vars_t5 = ['p_CV101', 'p_CV103', 'p_CV105', 'p_CV107', 'p_CV108', 'p_CV109', 'p_A']
p_def_a = lambda e: e['p_A'] == (e['p_CV101'] and e['p_CV103'] and e['p_CV105'] and not e['p_CV107'] and not e['p_CV108'] and not e['p_CV109'])
concl_t5 = lambda e: (not (e['p_CV107'] or e['p_CV108'] or e['p_CV109'])) if e['p_A'] else True

res_t5_tab = ProvadorDedutivoFormal.verificar_tabela_verdade("Teorema 5: Qualidade Assegurada Cat A", vars_t5, [p_def_a], concl_t5)

relatorio_t3_t5 = [
    {"Teorema": res_t3_tab['Teorema'], "Estados (2^n)": res_t3_tab['Total Estados (2^n)'], "Linhas Consistentes": res_t3_tab['Estados Válidos (Premissas True)'], "Status": res_t3_tab['Status']},
    {"Teorema": res_t4_tab['Teorema'], "Estados (2^n)": res_t4_tab['Total Estados (2^n)'], "Linhas Consistentes": res_t4_tab['Estados Válidos (Premissas True)'], "Status": res_t4_tab['Status']},
    {"Teorema": res_t5_tab['Teorema'], "Estados (2^n)": res_t5_tab['Total Estados (2^n)'], "Linhas Consistentes": res_t5_tab['Estados Válidos (Premissas True)'], "Status": res_t5_tab['Status']},
]
print("=== RESULTADOS DAS PROVAS FORMAIS (TEOREMAS 3, 4 E 5) ===")
print(formatar_tabela(relatorio_t3_t5))

assert res_t3_tab['Válido'] is True
assert res_t4_tab['Válido'] is True
assert res_t5_tab['Válido'] is True


## 4. Análise e Detecção de Falácias Formais Industriais
Identificação e bloqueio de raciocínios falaciosos na automação industrial:
1. **Falácia 1 (Afirmação do Consequente):** $c_{	ext{ALIM}} 
ightarrow p_{	ext{MOV201}} 
ot
dash p_{	ext{MOV201}} 
ightarrow c_{	ext{ALIM}}$.
2. **Falácia 2 (Negação do Antecedente):** $
eg p_{	ext{EMERG}} 
ot
dash c_{	ext{PERM}}$ (a ausência de emergência não garante permissivo se houver sobrecarga ou despressurização).


In [ ]:
# Falácia 1: Afirmação do Consequente na Esteira
vars_f1 = ['c_ALIM', 'p_MOV201']
prem_f1 = lambda e: (not e['c_ALIM']) or e['p_MOV201']
concl_f1 = lambda e: (not e['p_MOV201']) or e['c_ALIM']
res_f1 = ProvadorDedutivoFormal.verificar_tabela_verdade("Falácia 1: Afirmação do Consequente", vars_f1, [prem_f1], concl_f1)

# Falácia 2: Negação do Antecedente na Permissão Geral
vars_f2 = ['p_EMERG', 'p_PAL601', 'c_PERM']
prem_f2 = lambda e: e['c_PERM'] == (not e['p_EMERG'] and not e['p_PAL601'])
concl_f2 = lambda e: e['c_PERM'] if (not e['p_EMERG']) else True # Supor que NOT p_EMERG garante c_PERM (ignora PAL-601)
res_f2 = ProvadorDedutivoFormal.verificar_tabela_verdade("Falácia 2: Negação do Antecedente", vars_f2, [prem_f2], concl_f2)

relatorio_falacias = [
    {"Argumento": res_f1['Teorema'], "Status": res_f1['Status'], "Contraexemplos": len(res_f1['Contraexemplos'])},
    {"Argumento": res_f2['Teorema'], "Status": res_f2['Status'], "Contraexemplos": len(res_f2['Contraexemplos'])},
]
print("=== DETECÇÃO E DIAGNÓSTICO DE FALÁCIAS FORMAIS ===")
print(formatar_tabela(relatorio_falacias))

print("\nContraexemplos da Falácia 1:")
for ce in res_f1['Contraexemplos']:
    print(f"  -> {ce} (Esteira rodando sem alimentação ativa)")

print("\nContraexemplos da Falácia 2:")
for ce in res_f2['Contraexemplos']:
    print(f"  -> {ce} (Sem emergência acionada, mas bloqueado por baixa pressão pneumática)")

assert res_f1['Válido'] is False
assert res_f2['Válido'] is False
print("\n[OK] O provador formal detectou e barrou com rigor todas as falácias formais!")
